In [ ]:
import numpy as np #Importing required libraries
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

images = np.load("/content/mnistsemanticsegdataset/gridimages.npy") #Loading datasets
masks = np.load("/content/mnistsemanticsegdataset/gridmasks.npy")

print("Original dataset size:", images.shape) #Printing size of original ground dataset

images = images[:20000] #Only small portion of data is used as otherwise it is crashing my system
masks = masks[:20000]

print("Using subset:", images.shape) #Size of that subset of dataset

images = images.astype("float32") / 255.0
masks = np.expand_dims((masks > 0).astype("float32"), -1) #Adding channel dimension and converting it to binary as required by cnn

imgtrain, imgtest, labeltrain, labeltest = train_test_split( #splitting data as training and testing data
    images, masks, test_size=0.1, random_state=42
)

print("Train images:", imgtrain.shape) #Getting their sizes
print("Test images:", imgtest.shape)

def dicecoefficient(truemask, predmask, smooth=1e-6):
    truemaskflat = K.flatten(truemask) #Converting them to 1d vectors
    predmaskflat = K.flatten(predmask)
    intersection = K.sum(truemaskflat * predmaskflat) #Getting their overlapped area
    return (2. * intersection + smooth) / (K.sum(truemaskflat) + K.sum(predmaskflat) + smooth)

def diceloss(truemask, predmask): #Loss function which we need to minimise
    return 1 - dicecoefficient(truemask, predmask)

def buildunet(input_shape=(56, 56, 1)): #Defining training function
    inputs = layers.Input(shape=input_shape) #Getting input

    # Encoder
    convo1 = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs) #Applying first convolutional layer
    convo1 = layers.Conv2D(16, 3, activation='relu', padding='same')(convo1)
    pool1 = layers.MaxPooling2D()(convo1)

    convo2 = layers.Conv2D(32, 3, activation='relu', padding='same')(pool1) #Second convolutional layer
    convo2 = layers.Conv2D(32, 3, activation='relu', padding='same')(convo2)
    pool2 = layers.MaxPooling2D()(convo2)

    convo3 = layers.Conv2D(64, 3, activation='relu', padding='same')(pool2) #Third convolutional layer
    convo3 = layers.Conv2D(64, 3, activation='relu', padding='same')(convo3)
    pool3 = layers.MaxPooling2D()(convo3)

    #Bottleneck
    bot = layers.Conv2D(128, 3, activation='relu', padding='same')(pool3)#getting high level features
    bot = layers.Conv2D(128, 3, activation='relu', padding='same')(bot)

    # Decoder
    upsam3 = layers.UpSampling2D()(bot)  #Upsampling done to get back original size output
    upsam3 = layers.concatenate([upsam3, convo3])
    con4 = layers.Conv2D(64, 3, activation='relu', padding='same')(upsam3)
    con4 = layers.Conv2D(64, 3, activation='relu', padding='same')(con4)

    upsam2 = layers.UpSampling2D()(con4) #Upsampling done again
    upsam2 = layers.concatenate([upsam2, convo2])
    con5 = layers.Conv2D(32, 3, activation='relu', padding='same')(upsam2)
    con5 = layers.Conv2D(32, 3, activation='relu', padding='same')(con5)

    upsam1 = layers.UpSampling2D()(con5) #Last upsampling round
    upsam1 = layers.concatenate([upsam1, convo1])
    con6 = layers.Conv2D(16, 3, activation='relu', padding='same')(upsam1)
    con6 = layers.Conv2D(16, 3, activation='relu', padding='same')(con6)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(con6) #final output layer

    model = models.Model(inputs, outputs) #Linking input and output
    return model

model = buildunet() #Performing function and printing architechture in systematic manner
model.summary()

model.compile(optimizer=Adam(1e-4), #Difing optimizer , loss and metrice
              loss=dice_loss,
              metrics=[dice_coefficient])

history = model.fit(
    imgtrain, labeltrain,
    validation_data=(imgtest, labeltest),
    epochs=5,              # reduced for quick run
    batch_size=8,          # small batch to prevent session crashing
    verbose=1
)

testdice = model.evaluate(imgtest, labeltest, verbose=0)[1] #Getting dice coefficent
print(f"\n Test Dice Coefficient: {testdice:.4f}")

#Sample output results
n = 3
preds = model.predict(imgtest[:n])

#Binarize both ground truth and predictions for clarity
predsbin = (preds > 0.5).astype(np.uint8)
labeltestbin = (labeltest[:n] > 0.5).astype(np.uint8)

plt.figure(figsize=(12, 6))
for i in range(n):
    # Input
    plt.subplot(3, n, i + 1)
    plt.imshow(imgtest[i].squeeze(), cmap='gray')
    plt.title("Input")
    plt.axis('off')

    # Ground Truth
    plt.subplot(3, n, i + 1 + n)
    plt.imshow(labeltestbin[i].squeeze(), cmap='gray')
    plt.title("Ground Truth")
    plt.axis('off')

    # Prediction
    plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(predsbin[i].squeeze(), cmap='gray')
    plt.title("Predicted")
    plt.axis('off')

plt.tight_layout() #Displaying output
plt.show()
